<a href="https://colab.research.google.com/github/Varun-men/Foss-Hack-Ignivis/blob/main/environmental_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv("/content/Book 16(Sheet1).csv")

print(df.head())
print(df.columns)

   YEAR  Day  SolarRadiation  Temperature  Humidity  WindSpeed
0  2024    1            7.22        12.19     59.47       1.31
1  2024    2            8.77        12.34     56.57       1.35
2  2024    3            3.99        13.09     45.68       1.03
3  2024    4            2.09        11.26     60.43       1.00
4  2024    5            2.50        11.51     68.29       0.96
Index(['YEAR', 'Day', 'SolarRadiation', 'Temperature', 'Humidity',
       'WindSpeed'],
      dtype='object')


In [ ]:
import pandas as pd

df = pd.read_csv("Book 16(Sheet1).csv")


df['Temp_norm'] = (df['Temperature'] - df['Temperature'].min()) / (df['Temperature'].max() - df['Temperature'].min())
df['Hum_norm'] = (df['Humidity'] - df['Humidity'].min()) / (df['Humidity'].max() - df['Humidity'].min())
df['Wind_norm'] = (df['WindSpeed'] - df['WindSpeed'].min()) / (df['WindSpeed'].max() - df['WindSpeed'].min())

df['UV_Index'] = (
    (0.6 * df['Temp_norm']) -
    (0.3 * df['Hum_norm']) +
    (0.2 * df['Wind_norm'])
)


df['UV_Index'] = df['UV_Index'] * 11


df['UV_Index'] = df['UV_Index'].clip(0, 11)

print(df[['Temperature','Humidity','WindSpeed','UV_Index']].head())

   Temperature  Humidity  WindSpeed  UV_Index
0        12.19     59.47       1.31  5.406206
1        12.34     56.57       1.35  5.416014
2        13.09     45.68       1.03  5.452998
3        11.26     60.43       1.00  5.396724
4        11.51     68.29       0.96  5.374458


In [ ]:

df['Temp_n'] = (df['Temperature'] - df['Temperature'].min()) / (df['Temperature'].max() - df['Temperature'].min())
df['Hum_n'] = (df['Humidity'] - df['Humidity'].min()) / (df['Humidity'].max() - df['Humidity'].min())
df['UV_n'] = df['UV_Index'] / 11   # already 0–11 scale
df['Wind_n'] = (df['WindSpeed'] - df['WindSpeed'].min()) / (df['WindSpeed'].max() - df['WindSpeed'].min())


df['Environmental_Score'] = (
    (0.35 * df['Temp_n']) +
    (0.25 * df['Hum_n']) +
    (0.30 * df['UV_n']) -
    (0.20 * df['Wind_n'])
)


df['Environmental_Score'] = df['Environmental_Score'] * 100

print(df[['Environmental_Score']].head())

   Environmental_Score
0            53.095758
1            53.060330
2            52.943358
3            53.066768
4            53.195288


In [ ]:
def env_level(score):
    if score < 30:
        return "LOW 🟢"
    elif score < 70:
        return "MODERATE 🟡"
    else:
        return "HIGH 🔴"

df['Env_Level'] = df['Environmental_Score'].apply(env_level)

print(df[['Environmental_Score','Env_Level']].head())

   Environmental_Score   Env_Level
0            53.095758  MODERATE 🟡
1            53.060330  MODERATE 🟡
2            52.943358  MODERATE 🟡
3            53.066768  MODERATE 🟡
4            53.195288  MODERATE 🟡


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

features = ['Temperature','Humidity','WindSpeed','UV_Index']
X = df[features]
y = df['Environmental_Score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model_env = RandomForestRegressor()
model_env.fit(X_train, y_train)

print("Environmental model trained ✅")

Environmental model trained ✅


In [ ]:
sample = [[35, 60, 10, 8]]  # Temp, Humidity, Wind, UV

pred_score = model_env.predict(sample)[0]

print("Environmental Score:", round(pred_score,2))

Environmental Score: 54.09


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [ ]:
import joblib

joblib.dump(model_env, "environment_model.joblib")

# Load models
env_model = joblib.load("environment_model.joblib")


print("Models loaded successfully ✅")

Models loaded successfully ✅


In [ ]:
from google.colab import drive
drive.mount('/content/drive')